<a href="https://colab.research.google.com/github/shahad030/Final-Project--Bayan-bilingual-citizen-feedback/blob/main/Copy_of_Student_Lab1_Bilingual_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Clean the Messages | تنظيف الرسائل
**SDA-AIE-211 | Day 1 **

## القصة | The story
- 🇸🇦 وصلت إلى منصة **بيان** آلاف الرسائل من المواطنين، وفيها أرقام جوال وإيموجي وتطويل وأخطاء. قبل أي نموذج، يجب **تنظيف الرسائل بأمان** ثم **اختيار النموذج المناسب** للعربية والإنجليزية.
- 🇬🇧 The **Bayan** service received thousands of citizen messages full of phone numbers, emojis and noise. Before any model, we must **clean safely** and **pick the right model** for Arabic and English.

## الخطوات | Steps
| # | الخطوة | Step |
|---|---|---|
| 0 | تحميل البيانات | Load data |
| 1 | البحث عن العيوب | Find the defects |
| 2 | إخفاء البيانات الشخصية وتنظيف النص | Mask personal data and clean |
| 3 | تقسيم الجمل | Split sentences |
| 4 | مقارنة النماذج (Fertility) | Compare tokenizers |
| 5 | القرار وحفظ الملف | Decision and save |


In [ ]:
# ---- 0. Install libraries (run once per Colab session) ----
# datasets / accelerate / evaluate / seqeval -> later labs (training + evaluation)
# sentence-transformers / faiss-cpu          -> later labs (embeddings + similarity search)
# spacy                                      -> sentence segmentation in this lab
!pip -q install pandas numpy scikit-learn matplotlib transformers datasets accelerate evaluate seqeval sentence-transformers faiss-cpu spacy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


## Step 0 — تحميل البيانات | Load data

In [ ]:
# ---- 1. Imports, Google Drive and paths ----
from google.colab import drive
drive.mount('/content/drive')

import re, json, unicodedata, random
from pathlib import Path
import numpy as np
import pandas as pd


# Change ONLY this path if your folder is somewhere else
DATA_DIR = Path('/content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data')

# Fail early with a clear message if a file is missing
for name in ['bayan_feedback.csv', 'bayan_raw_sample.csv', 'pii_test_set.csv']:
    assert (DATA_DIR / name).exists(), f'Missing file: {DATA_DIR / name}'
print('All data files found.')

Mounted at /content/drive
All data files found.


In [ ]:
# ---- 2. Load the main dataset ----
feedback = pd.read_csv(DATA_DIR / 'bayan_feedback.csv')
print(feedback.shape)
print(feedback.columns.tolist())

# Quick health check: missing values and language balance
print(feedback[['text', 'language']].isna().sum())
print(feedback['language'].value_counts())

(12000, 11)
['feedback_id', 'citizen_id', 'text', 'language', 'dialect_region', 'topic', 'sentiment', 'channel', 'city', 'service_entity', 'created_day']
text        0
language    0
dtype: int64
language
en    6000
ar    6000
Name: count, dtype: int64


## Step 1 — البحث عن العيوب | Find the defects
- 🇸🇦 اقرأ الرسائل الخام، ثم شاهد عدد كل عيب.
- 🇬🇧 Read the raw messages, then see how often each defect appears.

In [ ]:
# ---- 3. Look at raw records ----
raw = pd.read_csv(DATA_DIR / 'bayan_raw_sample.csv')
display(raw[['feedback_id', 'text', 'language']].head(12))

display(raw[['feedback_id','text','language']].head(12))
# TODO: add six defects you actually observed in the sample.

defects = pd.DataFrame([
    {
        "Defect": "Tatweel / Elongation",
        "Example": "التطبيــــق",
        "Description": "Unnecessary character elongation in Arabic text."
    },
    {
        "Defect": "Diacritics (tashkeel) noise",
        "Example": "أحتِاج توضيح الرسومُ اُلإضافية في اِلِفاتورة (BF00087)",
        "Description": "Random harakat scattered through Arabic text (247 rows). The same word splits into several tokens."
    },
    {
        "Defect": "Letter flooding (repeated characters)",
        "Example": "تججججاوزت / السبتتتت (BF00098, BF00112)",
        "Description": "Letters repeated for emphasis (437 rows). Creates rare word forms and bloats the vocabulary."
    },
    {
        "Defect": "Personal data (PII) in text",
        "Example": "رقمي 0599447746 (BF00008), fahad.alharbi63@mail.example.sa (BF00272)",
        "Description": "Phone numbers (704 rows), national-ID-like numbers (295 rows) and emails (242 rows) sit inside the complaint text. Privacy risk, and the model could memorise them."
    },
    {
        "Defect": "Whitespace and punctuation irregularities",
        "Example": "The  street was cleaned (BF00030), ما يـصير كذا ؟؟؟ (BF00010)",
        "Description": "Double spaces and leading/trailing spaces (982 rows), plus runs of !!! or ؟؟؟ and spaces before them (2,366 rows). Same sentence, different surface forms."
    },
    {
        "Defect": "Exact duplicate texts",
        "Example": "Nothing has changed since my last message 😡 (BF00063)",
        "Description": "200 rows repeat another row's text exactly (256 after cleaning). Can leak between train and test."
    },
    {
        "Defect": "Label inconsistency",
        "Example": "BF00008: topic=billing but service_entity=permits; BF00215: sentiment=positive but text says هذا غير مقبول 😞",
        "Description": "topic and service_entity disagree in 1,739 rows (14.5%). Sentiment contradicts the text in about 200 rows, so the model learns from wrong targets."
    },

])

display(defects)

,feedback_id,text,language
0,BF05091,مرة أخرى، أريد تغيير موعدي يوم الجمعة. شكرا مق...,ar
1,BF01164,I missed my connection because the schedule sh...,en
2,BF00918,تذكير: يا أخي ما وصلني المبلغ المسترد 200 ريال...,ar
3,BF05630,مرة أخرى؟ كل ما في الثلاجة يفسد ولم يعطنا أحد ...,ar
4,BF03709,Question: The trip took 61 minutes instead of ...,en
5,BF05073,خبر جيد: تم خصم 75 ريال مرتين من فاتورتي، ومن ...,ar
6,BF06579,The garbage was not collected today. This is n...,en
7,BF09052,Really frustrated. I need help with my reques...,en
8,BF00811,محبط فعلا. أريد تغيير موعدي يوم الاثنين. أـــر...,ar
9,BF06013,للتوثيق، يعمل على الحاسوب ومش بيشتغل على الجوا...,ar


,feedback_id,text,language
0,BF05091,مرة أخرى، أريد تغيير موعدي يوم الجمعة. شكرا مق...,ar
1,BF01164,I missed my connection because the schedule sh...,en
2,BF00918,تذكير: يا أخي ما وصلني المبلغ المسترد 200 ريال...,ar
3,BF05630,مرة أخرى؟ كل ما في الثلاجة يفسد ولم يعطنا أحد ...,ar
4,BF03709,Question: The trip took 61 minutes instead of ...,en
5,BF05073,خبر جيد: تم خصم 75 ريال مرتين من فاتورتي، ومن ...,ar
6,BF06579,The garbage was not collected today. This is n...,en
7,BF09052,Really frustrated. I need help with my reques...,en
8,BF00811,محبط فعلا. أريد تغيير موعدي يوم الاثنين. أـــر...,ar
9,BF06013,للتوثيق، يعمل على الحاسوب ومش بيشتغل على الجوا...,ar


,Defect,Example,Description
0,Tatweel / Elongation,التطبيــــق,Unnecessary character elongation in Arabic text.
1,Diacritics (tashkeel) noise,أحتِاج توضيح الرسومُ اُلإضافية في اِلِفاتورة (...,Random harakat scattered through Arabic text (...
2,Letter flooding (repeated characters),"تججججاوزت / السبتتتت (BF00098, BF00112)",Letters repeated for emphasis (437 rows). Crea...
3,Personal data (PII) in text,"رقمي 0599447746 (BF00008), fahad.alharbi63@mai...","Phone numbers (704 rows), national-ID-like num..."
4,Whitespace and punctuation irregularities,"The street was cleaned (BF00030), ما يـصير كذ...",Double spaces and leading/trailing spaces (982...
5,Exact duplicate texts,Nothing has changed since my last message 😡 (B...,200 rows repeat another row's text exactly (25...
6,Label inconsistency,BF00008: topic=billing but service_entity=perm...,"topic and service_entity disagree in 1,739 row..."


## Step 2 — إخفاء البيانات الشخصية وتنظيف النص | Mask personal data and clean
- 🇸🇦 **القاعدة الذهبية:** أخفِ البيانات الشخصية (PII) **أولًا**، ثم نظّف النص.
- 🇬🇧 **Golden rule:** mask personal data (PII) **first**, then clean the text.
- 🔒 الدالة `mask_pii` فيها تعابير نمطية (Regex) معقدة → **للقراءة فقط** /

Mask PII → Unicode NFC Normalization → Remove Tatweel → Normalize Whitespace → Strip Extra Spaces → Normalized Text  → Test (PII masking)

In [ ]:
# Version of the preprocessing pipeline
PREPROCESSING_VERSION = "bayan-preproc-v1"


# Function to hide personal information (PII)
def mask_pii(text):

    text = str(text)

    # Regular Expression for email addresses:
    # [A-Za-z0-9._%+-]+ = characters before @
    # @ = the @ symbol
    # [A-Za-z0-9.-]+ = email domain
    # \. = dot before the extension
    # [A-Za-z]{2,} = extension with at least 2 letters
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        '[EMAIL]', text
    )

    # Regular Expression for Saudi mobile numbers:
    # 05 = Saudi mobile prefix
    # (?:[\s-]?\d){8} = exactly 8 digits, each may have a space or hyphen before it
    #                   (so 055-123-4567 and 055 123 4567 are both matched)
    # (?<!\d) and (?!\d) prevent matching part of a longer number

    #TODO: Add regular expression for Saudi mobile numbers
    text = re.sub(
        r'(?<!\d)[0٠][5٥](?:[\s-]?\d){8}(?!\d)',
        '[PHONE]', text
    )


    # Regular Expression for Saudi national ID:
    # [12] = starts with 1 or 2
    # \d{9} = followed by 9 digits
    # (?<!\d) and (?!\d) prevent matching part of a longer number

    #TODO: Add regular expression for Saudi national ID
    text = re.sub(
        r'(?<!\d)[12١٢]\d{9}(?!\d)',
        '[NATIONAL_ID]', text
    )


    return text


# Function to normalize Arabic and English text
def normalize(text):

    # Convert text to Unicode NFC format
    text = unicodedata.normalize('NFC', str(text))

    # Remove Arabic Tatweel (ـ)
    text = text.replace('ـ', '')

    # Replace multiple spaces with one space
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces at the beginning and end
    return text.strip()


# Function to apply preprocessing in the correct order
def preprocess(text):

    # Step 1: Hide personal information FIRST
    text = mask_pii(text)

    # Step 2: Normalize the text
    text = normalize(text)

    return text


# Load the PII test dataset
pii = pd.read_csv(
    "/content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data/pii_test_set.csv"
)

# Test whether all expected PII values are masked
passed = [
    str(r.expected_value) not in mask_pii(r.text) #check if the value available or not should return true
    for r in pii.itertuples()
]

# Display test results
print(f"PII masking tests: {sum(passed)}/{len(passed)}")

# Stop if any test fails
assert all(passed), "Masking test failed; check the regular expressions."


# TODO: Test phone numbers with different formats
test_numbers = [
    "0551234567",                    # plain
    "055-123-4567",                  # hyphens
    "055 123 4567",                  # spaces
    "٠٥١-٩٣٦-٢١٤٩",                  # Arabic-Indic digits
    "call me on 0551234567 please",  # inside a sentence
    "05512345678",                   # 11 digits: must NOT be masked
    "0412345678",                    # not a 05 prefix: must NOT be masked
]



for number in test_numbers:
    print(number, "→", mask_pii(number))

PII masking tests: 60/60
0551234567 → [PHONE]
055-123-4567 → [PHONE]
055 123 4567 → [PHONE]
٠٥١-٩٣٦-٢١٤٩ → [PHONE]
call me on 0551234567 please → call me on [PHONE] please
05512345678 → 05512345678
0412345678 → 0412345678


In [ ]:
feedback['text_clean'] = feedback['text'].map(preprocess)

## Step 3 — تقسيم الجمل | Segmentation
- 🇸🇦 أحيانًا تحتوي الرسالة على أكثر من جملة. نقسّمها عند علامات النهاية.
- 🇬🇧 A message can hold several sentences. We split at sentence-ending marks.

In [ ]:
def split_sentences(text):
    parts = re.split(r'(?<=[.!?؟])\s+', preprocess(text))   # نقسم بعد . ! ? ؟  \s أي مسافة بيضاء (Whitespace), + تكرار مرة واحدة او اكثر
    return [p for p in parts if p] #p موجودة وليست فارغة

print(split_sentences('الخدمة بطيئة جدا. هل يمكن المساعدة؟ شكرا'))

['الخدمة بطيئة جدا.', 'هل يمكن المساعدة؟', 'شكرا']


## Step 4 — مقارنة النماذج | Compare tokenizers
- 🇸🇦 **Fertility** = عدد القطع (tokens) ÷ عدد الكلمات. كلما قلّ الرقم كان النموذج "يفهم" اللغة بكفاءة أكبر.
- 🇬🇧 **Fertility** = tokens ÷ words. Lower means the model handles the language more efficiently.
- **p95:** طول 95٪ من الرسائل (نستخدمه لاحقًا لاختيار أقصى طول) / length that 95% of messages fit in. طول 95% من الرسائل بوحدة Tokens
- 🔒 الكود للقراءة فقط، شغّله وانظر الجدول / read-only — just run it and read the table.

In [ ]:
# استيراد المكتبات اللازمة لتفادي أخطاء عدم التعريف
# Import necessary libraries to prevent NameErrors if run out of order
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

# التحقق من وجود عمود النصوص المنظفة وتجهيزه إذا كان مفقوداً
# Ensure 'text_clean' exists; if not, create it using our preprocessing pipeline
if 'text_clean' not in feedback.columns:
    feedback['text_clean'] = feedback['text'].map(preprocess)

# تحديد النماذج التي نريد مقارنة Tokenizers الخاصة بها
# Define the models we want to compare

candidates = {
    'xlm-roberta-base': 'XLM-R',
    'bert-base-multilingual-cased': 'mBERT',
    'CAMeL-Lab/bert-base-arabic-camelbert-mix': 'CAMeLBERT',
}


# إنشاء قائمة فارغة لتخزين نتائج المقارنة
# Create an empty list to store comparison results

rows = []


# المرور على كل نموذج من النماذج الثلاثة
# Loop through each of the three models

for ckpt, label in candidates.items(): #name of the model and shortcut

    # تحميل Tokenizer الخاص بالنموذج الحالي
    # Load the tokenizer for the current model

    tok = AutoTokenizer.from_pretrained(ckpt)


    # اختبار النموذج على اللغتين العربية والإنجليزية
    # Test the tokenizer on Arabic and English

    for lang in ['ar', 'en']:

        # اختيار أول 300 رسالة من اللغة الحالية
        # Select the first 300 messages in the current language

        texts = feedback.loc[
            feedback.language == lang, 'text_clean'
        ].head(300).tolist()


        # حساب إجمالي عدد Tokens في الرسائل
        # Count the total number of tokens across all messages

        pieces = sum(len(tok.tokenize(t)) for t in texts) #tokens of the 300 message


        # حساب إجمالي عدد الكلمات في الرسائل
        # Count the total number of words across all messages

        words = sum(len(t.split()) for t in texts) #words in the 300 message


        # حساب عدد Tokens في كل رسالة
        # Calculate the number of tokens in each message

        lengths = [len(tok(t)['input_ids']) for t in texts] #from each token ID


        # إضافة نتائج النموذج واللغة إلى القائمة
        # Add the model and language results to the list

        rows.append({
            'model': label,
            'language': lang,

            # Fertility = إجمالي Tokens ÷ إجمالي الكلمات
            # Average number of tokens per word

            'fertility': round(pieces / words, 2), #round 2

            # حساب المئين الخامس والتسعين لطول الرسائل
            # Calculate the 95th percentile of message lengths

            'p95_length': int(np.percentile(lengths, 95))
        })


# تحويل قائمة النتائج إلى DataFrame لعرضها كجدول
# Convert the results list into a DataFrame

audit = pd.DataFrame(rows)


# عرض جدول المقارنة
# Display the comparison table

display(audit)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

,model,language,fertility,p95_length
0,XLM-R,ar,1.71,41
1,XLM-R,en,1.37,41
2,mBERT,ar,2.29,56
3,mBERT,en,1.41,42
4,CAMeLBERT,ar,1.44,35
5,CAMeLBERT,en,2.51,73


## Step 5 — القرار وحفظ الملف | Decision and save
- 🇸🇦 اختر **نموذجًا للعربية** و**نموذجًا للغتين**، واكتب سببك مستندًا إلى جدول Step 4.
- 🇬🇧 Choose an **Arabic model** and a **bilingual model**, and justify with the Step 4 table.

In [ ]:
import json
from pathlib import Path

# Choose checkpoint(s) using measured fertility, p95 length, and language coverage
decision = {
    'arabic_checkpoint': 'CAMeL-Lab/bert-base-arabic-camelbert-mix',
    'bilingual_checkpoint': 'xlm-roberta-base',
    'evidence': (
        'CAMeLBERT has the best Arabic efficiency (fertility 1.44, p95 35) but is poor on English '
        '(fertility 2.51, p95 73). XLM-R is the most balanced across both languages '
        '(fertility ar 1.71 / en 1.37, p95 41 / 41) and beats mBERT on Arabic '
        '(mBERT: 2.29, p95 56). Use XLM-R as the single bilingual model with max_length 64, '
        'which covers p95 with headroom.'
    )
}

# Print the decision as formatted JSON
print(json.dumps(decision, indent=2, ensure_ascii=False))

# حفظ الملفات للابات القادمة | Save files for the next labs
# Local writable folder (the Drive folder is read-only)
ARTIFACT_DIR = Path('/content/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACT_DIR / 'lab1_decision.json', 'w', encoding='utf-8') as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)


print('✅ Saved decision + clean data')

{
  "arabic_checkpoint": "CAMeL-Lab/bert-base-arabic-camelbert-mix",
  "bilingual_checkpoint": "xlm-roberta-base",
  "evidence": "CAMeLBERT has the best Arabic efficiency (fertility 1.44, p95 35) but is poor on English (fertility 2.51, p95 73). XLM-R is the most balanced across both languages (fertility ar 1.71 / en 1.37, p95 41 / 41) and beats mBERT on Arabic (mBERT: 2.29, p95 56). Use XLM-R as the single bilingual model with max_length 64, which covers p95 with headroom."
}
✅ Saved decision + clean data


## Thinking challenges
1. Why preserve emoji for sentiment?
2. What breaks if train and serving use different normalisers?
3. Defend one multilingual tokenizer versus separate Arabic/English models using your audit.